# CARBAMYLATION ANALYSIS TOOL

This Jupyter notebook enables selecting and curating protein atomic structures for the analysis of their lysines. The aim, is to identify the subset of lysines that may be carbamylated. The pipeline is divided in the following steps.

* 1. [Data Gathering](#gather)
  - 1.1. [Mine UNIPROT database](#uniprot)
  - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)

Further explanations about each of these steps are provided in the individual sections below.
Now, start by executing the cell below (press Shift+ENTER keys) to imports all the Python packages required.

In [1]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox

# atomic structures manipulation
import biobox as bb

#carbamylation analysis procedures
import postprocessing as postproc
from uniprot import Uniprot
from protein import PDB
from measure import Measure

C:\Users\xdzl45\Anaconda3\lib\site-packages\blessed\terminal.py:186: UserWarning: Failed to setupterm(kind='xterm-color'): Could not find terminal xterm-color
  warnings.warn(msg)


## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database  <a class="anchor" id="uniprot"></a>

Load data from UNIPROT database. The commands below can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`).

In [2]:
UP = Uniprot()
#UP.get_protein_data("P09167") # example: load structures for a single UNIPROT
UP.from_csv_file("inputs\\input_codes_4.csv") # add structures from a .csv file
UP.df

.csv file successfully opened


,Uniprot Entry,PDB Code,Method,Resolution,Chains
0,P62987,2MBH,NMR,-,A
1,P04406,1U8F,X-ray,1.75,O/P/Q/R
2,Q9HB71,AF-Q9HB71-F1-model_v1,Predicted,NaN,NaN
3,P15311,1NI2,X-ray,2.30,A/B


### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

Load and curate all proteins described in a Uniprot DataFrame (`Uniprot.df`). For each entry in the database:
* raw files are downloaded from the PDB in a working folder (defined by `PDB.raw_dir`)
* each raw file is cleaned: heteroatoms, hydrogens and non-protein and non-ion atoms removed, non-standard aminoacids are amended
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different folder (defined by `PDB.curated_dir`)

This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [3]:
pdb = PDB()
pdb.gather_proteins(UP.df)


UNIPROT: P62987, PDB: 2MBH
> downloading PDB 2MBH
> downloading FASTA for 2MBH
> Alternate model(s) found. Splitting...
>> Patching failed for conformer 0: large gap detected (21 residues)
>> Patching failed for conformer 1: large gap detected (21 residues)
>> Patching failed for conformer 2: large gap detected (21 residues)
>> Patching failed for conformer 3: large gap detected (21 residues)
>> Patching failed for conformer 4: large gap detected (21 residues)
>> Patching failed for conformer 5: large gap detected (21 residues)
>> Patching failed for conformer 6: large gap detected (21 residues)
>> Patching failed for conformer 7: large gap detected (21 residues)
>> Patching failed for conformer 8: large gap detected (21 residues)
>> Patching failed for conformer 9: large gap detected (21 residues)
>> Patching failed for conformer 10: large gap detected (21 residues)
>> Patching failed for conformer 11: large gap detected (21 residues)
>> Patching failed for conformer 12: large gap de

openf___224_> Open           ${LIB}/par.lib

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1011015     987.319     0.964

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1012615     988.882     0.966

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1016595     992.769     0.970

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1018595     994.722     0.971

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1021595     997.651     0.974

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1026095    1002.046     0.979

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1031667    1007.487     0.984

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1034467    1010.222     0.987

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1038667    1014.323     0.991

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      1044967    1020.


Dynamically allocated memory at  amaxrestraints [B,KiB,MiB]:      4094256    3998.297     3.905

Dynamically allocated memory at  amaxrestraints [B,KiB,MiB]:      4356400    4254.297     4.155

Dynamically allocated memory at  amaxrestraints [B,KiB,MiB]:      4880688    4766.297     4.655
make_re_422_> Number of previous, current restraints         :    12268    19747
make_re_423_> Number of previous, current selected restraints:    11924    19403
make_re_417_> Restraint type to be calculated:  DISTANCE

Dynamically allocated memory at  amaxrestraints [B,KiB,MiB]:      5142832    5022.297     4.905
make_re_422_> Number of previous, current restraints         :    19747    27346
make_re_423_> Number of previous, current selected restraints:    19403    27002
make_re_417_> Restraint type to be calculated:  DISTANCE
make_re_422_> Number of previous, current restraints         :    27346    32678
make_re_423_> Number of previous, current selected restraints:    27002    32334
make_re_417_


Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      7913518    7728.045     7.547

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      7928830    7742.998     7.562

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      7939074    7753.002     7.571

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      7977834    7790.854     7.608

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      8371618    8175.408     7.984

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      8394586    8197.838     8.006

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      8409950    8212.842     8.020

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      8468090    8269.619     8.076

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      9058766    8846.451     8.639

Dynamically allocated memory at amaxgroup_restr [B,KiB,MiB]:      9093218    8880.096     8.672

Dynamically allocated memory 



report______> Distribution of short non-bonded contacts:


DISTANCE1:  0.00 2.10 2.20 2.30 2.40 2.50 2.60 2.70 2.80 2.90 3.00 3.10 3.20 3.30 3.40
DISTANCE2:  2.10 2.20 2.30 2.40 2.50 2.60 2.70 2.80 2.90 3.00 3.10 3.20 3.30 3.40 3.50
FREQUENCY:     0    0    0    0    0   11   32   76  164  183  232  278  360  354  409


<< end of ENERGY.
openf___224_> Open           1U8F.B99990001.pdb
wrpdb___568_> Residues, atoms, selected atoms:      333     2523     2523

>> Summary of successfully produced models:
Filename                          molpdf     DOPE score    GA341 score
----------------------------------------------------------------------
1U8F.B99990001.pdb            1914.53333   -37930.83203        1.00000

>> modelling missing residues
openf___224_> Open           $(LIB)/restyp.lib
openf___224_> Open           ${MODINSTALL10v2}/modlib/resgrp.lib
rdresgr_266_> Number of residue groups:        2
openf___224_> Open           ${MODINSTALL10v2}/modlib/sstruc.lib

Dynamically allocate

...

openf___224_> Open           ${MODINSTALL10v2}/modlib/mnch3.lib
openf___224_> Open           ${MODINSTALL10v2}/modlib/xs4.mat
rdrrwgh_268_> Number of residue types:       21
openf___224_> Open           alignment.seg

Dynamically allocated memory at   amaxalignment [B,KiB,MiB]:      5535475    5405.737     5.279

Dynamically allocated memory at   amaxalignment [B,KiB,MiB]:      5536925    5407.153     5.280

Dynamically allocated memory at   amaxalignment [B,KiB,MiB]:      5539825    5409.985     5.283

Dynamically allocated memory at   amaxalignment [B,KiB,MiB]:      5545625    5415.649     5.289

Dynamically allocated memory at    amaxsequence [B,KiB,MiB]:      5546805    5416.802     5.290

Read the alignment from file       : alignment.seg

Total number of alignment positions:   296

  #  Code        #_Res #_Segm PDB_code    Name
-------------------------------------------------------------------------------
  1  result\cu     296      1 result\cura
openf___224_> Open           ali

openf___224_> Open           ${LIB}/par.lib

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6268802    6121.877     5.978

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6270402    6123.439     5.980

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6274382    6127.326     5.984

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6276382    6129.279     5.986

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6279382    6132.209     5.988

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6283882    6136.604     5.993

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6289454    6142.045     5.998

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6292254    6144.779     6.001

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6296454    6148.881     6.005

Dynamically allocated memory at  amaxparameters [B,KiB,MiB]:      6302754    6155.

getdata_643_> Protein accepted:  result\curated\tmp\chainB
getdata_289_> Proteins (all/accepted):        1        1
make_re_422_> Number of previous, current restraints         :    11907    12012
make_re_423_> Number of previous, current selected restraints:    11598    11703
make_re_417_> Restraint type to be calculated:  chi4_dihedral

Dynamically allocated memory at   amaxstructure [B,KiB,MiB]:      8335645    8140.278     7.949
openf___224_> Open           ${MODINSTALL10v2}/modlib/chi1234.bin
openf___224_> Open           ${MODINSTALL10v2}/modlib/chi4.mdt
mdtrsr__446W> A potential that relies on one protein is used, yet you have at
              least one known structure available. MDT, not library, potential is used.
getdata_643_> Protein accepted:  result\curated\tmp\chainB
getdata_289_> Proteins (all/accepted):        1        1
make_re_422_> Number of previous, current restraints         :    12012    12059
make_re_423_> Number of previous, current selected restraints:    11703



>> ENERGY; Differences between the model's features and restraints:
Number of all residues in MODEL                   :      296
Number of all, selected real atoms                :     2466    2466
Number of all, selected pseudo atoms              :        0       0
Number of all static, selected restraints         :    29440   29440
COVALENT_CYS                                      :        F
NONBONDED_SEL_ATOMS                               :        1
Number of non-bonded pairs (excluding 1-2,1-3,1-4):   454506
Dynamic pairs routine                             : 1, NATM x NATM double loop
Atomic shift for contacts update (UPDATE_DYNAMIC) :    0.390
LENNARD_JONES_SWITCH                              :    6.500   7.500
COULOMB_JONES_SWITCH                              :    6.500   7.500
RESIDUE_SPAN_RANGE                                :        1    9999
NLOGN_USE                                         :       15
CONTACT_SHELL                                     :   15.000
DYNAMIC_PA


DISTANCE1:  0.00 2.10 2.20 2.30 2.40 2.50 2.60 2.70 2.80 2.90 3.00 3.10 3.20 3.30 3.40
DISTANCE2:  2.10 2.20 2.30 2.40 2.50 2.60 2.70 2.80 2.90 3.00 3.10 3.20 3.30 3.40 3.50
FREQUENCY:     0    0    0    0    0    4   27   86  132  194  231  269  335  367  412


<< end of ENERGY.
openf___224_> Open           1NI2.B99990001.pdb
wrpdb___568_> Residues, atoms, selected atoms:      296     2466     2466

>> Summary of successfully produced models:
Filename                          molpdf     DOPE score    GA341 score
----------------------------------------------------------------------
1NI2.B99990001.pdb            1536.10693   -35255.11719        1.00000



Show the DataFrame reporting on structures successfully downloaded and curated.

In [4]:
pdb.df

,Uniprot Entry,PDB Code,Method,Resolution,Chains
0,P04406,1U8F,X-ray,1.75,O/P/Q/R
1,Q9HB71,AF-Q9HB71-F1-model_v1,Predicted,NaN,A
2,P15311,1NI2,X-ray,2.30,A/B


### 2. Feature Extraction  <a class="anchor" id="measure"></a>

For all structures correctly curated in the previous step, calculate pKa and Solvent Accessible Surface Area (SASA).

<div class="alert alert-warning">
<b>Warning:</b> calculations typically take ~1 min per curated protein structure (depending on their size and number of lysines).
</div>

In [5]:
M = Measure(pdb.df, "result")
M.measure_dataframe()

#TODO: Measure object saves DataFrame in a csv within its designated results folder


# UNIPROT: P04406 PDB: 1U8F, chain(s): O P Q R

> File: result\curated\1U8F-alt1A.pdb
>> 104 lysines of interest found
>> evaluating pKa...
>> evaluating sasa...

> File: result\curated\1U8F-alt1B.pdb
>> 104 lysines of interest found
>> evaluating pKa...
>> evaluating sasa...

# UNIPROT: Q9HB71 PDB: AF-Q9HB71-F1-model_v1, chain(s): A

> File: result\curated\AF-Q9HB71-F1-model_v1.pdb
>> 33 lysines of interest found
>> evaluating pKa...
>> evaluating sasa...

# UNIPROT: P15311 PDB: 1NI2, chain(s): A B

> File: result\curated\1NI2-alt-1.pdb
>> 62 lysines of interest found
>> evaluating pKa...
>> evaluating sasa...


Print the final measures output, sorted by UNIPROT entry first, and then by resid.

In [9]:
result = M.df.dropna()
result.sort_values(['Uniprot Entry', 'Resid'])

,Uniprot Entry,PDB Code,Method,Resolution,Chain,Resid,pKa,sasa
0,P04406,result\curated\1U8F-alt1A,X-ray,1.75,O,3,10.48,82.738980
26,P04406,result\curated\1U8F-alt1A,X-ray,1.75,P,3,10.65,54.270751
52,P04406,result\curated\1U8F-alt1A,X-ray,1.75,Q,3,10.22,52.025331
78,P04406,result\curated\1U8F-alt1A,X-ray,1.75,R,3,10.45,90.119727
104,P04406,result\curated\1U8F-alt1B,X-ray,1.75,O,3,10.67,74.643980
130,P04406,result\curated\1U8F-alt1B,X-ray,1.75,P,3,10.45,84.062703
156,P04406,result\curated\1U8F-alt1B,X-ray,1.75,Q,3,10.28,81.608956
182,P04406,result\curated\1U8F-alt1B,X-ray,1.75,R,3,10.45,90.119727
1,P04406,result\curated\1U8F-alt1A,X-ray,1.75,O,5,10.27,39.882028
27,P04406,result\curated\1U8F-alt1A,X-ray,1.75,P,5,10.93,47.108967


### 3. Data Analysis  <a class="anchor" id="analysis"></a>

The Analysis class is resonsible of interpreting data stored in the DataFrame `Measures.df`. Data is saved in `Analysis.df` and  automatically aggregated in `Analysis.df_aggregate`. Aggregation involves calculating min, max, mean and standard deviation for each feature (a single lysine may features multiple measurements in the input dataframe).
Here, the ground truth (i.e. known carbamylated lysines) can also be loaded. Ground truth is appended as an extra column to `Analysis.df` and `Analysis.df_aggregate`.

Start by initializing `Analysis` either via a previously saved logfile, or from a DataFrame in a `Measure` instance.

In [ ]:
if True: # is a Measures instance was produced above
    df = M.df
else: # if a logfile storing a Measures DataFrame calculated in a previous session 
    df = pd.read_csv("results\\results_2608\\results.csv")

#TODO 1: aggregate data according to the selection below (extract mean and std of each measure)
#entries = result.drop_duplicates(subset=["Uniprot Entry", "Resid"])[["Uniprot Entry", "Resid"]]
#entries
#
#TODO 2: load experimental evidence (ground truth, i.e. what is carbamylated) into an Analysis class.
#        The class stores both aggregated and raw data. Ground truth appended as extra column.


# A = Analysis(df)

Plot data on interactive plot. Gray points represent the aggregated data, featuring one point per uniprot code. Clicking on any point will show additional points in red. These points are the measures performed on every protein structure associated with the uniprot code selected by clicking.

In [ ]:
#ISSUE: hovering always looks as if the aggregate plot is selected, although focus items are clicked

df_data = M.df

def update_point(trace, points, selector):
          
    #print(trace, points)    
    
    # only work with gray points
    if len(points.point_inds) == 0:
        return  
    
    # gather uniprot and chain ID from full dataset clicked point
    idx = points.point_inds[0]
    my_uniprot = df_data.loc[idx, "Uniprot Entry"]
    my_resid = df_data.loc[idx, "Resid"]
    my_label = "%s(%i)"%(my_uniprot, my_resid)
    df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(my_uniprot, my_resid))

    #plot data associated with selection
    #labels = ["UNIPROT: %s<br>resid: %i"%(df_query["Uniprot Entry"].values[i], df_query["resid"].values[i]) for i in range(len(df_query))]
    labels = ["PDB: %s"%(df_query["PDB Code"].values[i]) for i in range(len(df_query))]
    if len(f.data)>1:
        f.data = [f.data[0]]

    f.add_scatter(x=df_query["sasa"], y=df_query["pKa"],
                  mode='markers', showlegend=False, name=my_label,
                  text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}')

    
    #identify clicked point and mark it differently (e.g. different colour or size)
    #c = list(scatter.marker.color)
    #for i in points.point_inds:
    #    c[i] = '#ffffff'
    #    with f.batch_update():
    #        scatter.marker.color = c

# create scatter plot
labels = ["UNIPROT: %s<br>resid: %i"%(df_data["Uniprot Entry"].values[i], df_data ["Resid"].values[i]) for i in range(len(df_data))]
f = go.FigureWidget([go.Scatter(x=df_data["sasa"], y=df_data["pKa"],
                                mode='markers', name="aggregate", showlegend=False, opacity=0.75,
                                text = labels, hovertemplate='%{text}<br>SASA: %{x:.2f}<br>pKa: %{y:.2f}'
                               )
                    ])

# add labels
f.update_layout(
    xaxis_title="SASA (A2)",
    yaxis_title="pKa")

#set axes properties
f.update_xaxes(range=[0, 130])
f.update_yaxes(range=[-2, 20])
f.update_xaxes(showspikes=True)
f.update_yaxes(showspikes=True)

#parameterize colours and triggers
scatter = f.data[0]
colors = ['#aaaaaa'] * len(df_data)
scatter.marker.color = colors
f.layout.hovermode = 'closest'
scatter.on_click(update_point)
f

Select desired uniprot entry and resid from the logfile, based on observation of the plot above.

In [ ]:
uniprot = "P15531" #insert uniprot code here
resid = 85 #insert resid here

Display all lysine entries matching the uniprot code and resid provided above, and provide statistics(this is useful if more than one lysine conformation was analysed).

In [ ]:
df_query = df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))
pKa = df_query["pKa"]
sasa = df_query["sasa"]
print("uniprot %s, resid %s\n%s measures\npKa %4.2f pm %4.2f [%4.2f %4.2f]\nsasa %4.2f pm %4.2f [%4.2f %4.2f] A2"%
      (uniprot, resid, len(pKa),
       np.mean(pKa), np.std(pKa), np.min(pKa), np.max(pKa),
       np.mean(sasa), np.std(sasa), np.min(sasa), np.max(sasa)))


df.query('`Uniprot Entry` == "%s" and resid == %s'%(uniprot, resid))

Select index of an entry of interest, to be visualized in 3D

In [ ]:
idx = 8699

Display the structure of interest

In [ ]:
resid = df.loc[idx, "resid"]
chain = df.loc[idx, "chain"]
PDB = "%s%s%s.pdb"%(folder_path, os.sep, df.loc[idx, "PDB Code"])
print("file: %s, chain: %s, resid: %s"%(PDB, chain, resid))

#u = mda.Universe(PDB)
#protein = u.select_atoms('protein')
#view = nv.show_mdanalysis(protein)
view

In [ ]:
# SCATCH AREA, EXAMPLE CODES FOR DIFFERENT USAGES #

# example: show data visible through hovering over points
#
#fig = px.scatter(df_datae, x="sasa", y="pKa", opacity=0.5,
#                 #error_x="sasa stdev", error_y="pKa stdev",
#                 hover_data=["PDB Code", "resid", "chain", "Uniprot Entry"])
#fig.show()

In [ ]:
import plotly.graph_objects as go
from ipywidgets import HBox, VBox

import numpy as np
np.random.seed(1)

x = np.random.rand(100)
y = np.random.rand(100)

e = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])
g = go.FigureWidget([go.Scatter(x=x, y=y, mode='markers')])#, go.Scatter(x=x, y=y, mode='markers')])

scattere = e.data[0]
colors = ['#a3a7e4'] * 100
scattere.marker.color = colors
scattere.marker.size = [10] * 100
e.layout.hovermode = 'closest'

scatterg = g.data[0]
colors = ['#a3a7e4'] * 100
scatterg.marker.color = colors
scatterg.marker.size = [10] * 100
g.layout.hovermode = 'closest'

# create our callback function
def update_pointe(trace, points, selector):
    c = list(scattere.marker.color)
    s = list(scattere.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with e.batch_update():
            scattere.marker.color = c
            scattere.marker.size = s
            
# create our callback function
def update_pointg(trace, points, selector):
    c = list(scatterg.marker.color)
    s = list(scatterg.marker.size)
    for i in points.point_inds:
        c[i] = '#bae2be'
        s[i] = 20
        with g.batch_update():
            scatterg.marker.color = c
            scatterg.marker.size = s



scattere.on_click(update_pointg)
scatterg.on_click(update_pointe)


VBox([e,g])

In [ ]:
def set_opacity(opacity, size):
    scattere.marker.opacity = opacity
    scattere.marker.size = size

from ipywidgets import interactive
opacity_slider = interactive(set_opacity,
                            opacity=(0.0, 1.0, 0.01),
                            size=(1,10,0.25))


opacity_slider.children[0].layout.width = '400px'
opacity_slider.children[1].layout.width = '400px'

VBox([e,g, opacity_slider])